# Remaining Useful Life Prediction for EV Batteries

**Machine Learning Project — Based on Swain et al., IEEE Access 2024**

This notebook loads real battery data from a CSV file that contains realistic missing values. The missing values are identified, visualised, and corrected using mean imputation before training the machine learning models.

**Battery Behaviour**

| Battery | Behaviour |
|---|---|
| B0005 | Fast aging — steep capacity loss |
| B0006 | Slow aging — very gradual decay |
| B0007 | Unstable — random stress event dips |
| B0018 | Temperature-sensitive — wavy degradation |

**Models:** Random Forest · SVR · Gradient Boosting

**Dataset file:** `ev_battery_data_with_missing.csv` — 10,000 rows, 18 columns

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import f_oneway

np.random.seed(55)
print('Libraries imported.')

## 2. Load Raw Dataset

The CSV file contains real sensor data with missing values caused by:
- Sensor dropouts (random single-row gaps)
- Equipment calibration periods (block gaps in impedance readings)
- Correlated failures (temperature charge and discharge sensors sharing a fault)

We load the raw file first and inspect it before any cleaning.

In [ ]:
# Load the raw CSV — missing values will appear as NaN
df_raw = pd.read_csv('ev_battery_data_with_missing.csv')

print(f'Rows loaded   : {df_raw.shape[0]}')
print(f'Columns       : {df_raw.shape[1]}')
print()
print('Samples per battery:')
print(df_raw.groupby('battery_id')['cycle'].count())
print()
print('First 4 rows (NaN = missing):')
df_raw.head(4)

## 3. Identify Missing Values

Before fixing anything, we count and display exactly where the missing values are. This step is important — it shows which sensors had problems and how serious the gaps are.

In [ ]:
# Count missing values in each column
missing_counts = df_raw.isnull().sum()
missing_pct    = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)

# Build a clear summary table
miss_summary = pd.DataFrame({
    'Missing Count'   : missing_counts,
    'Missing Percent' : missing_pct
})
miss_summary = miss_summary[miss_summary['Missing Count'] > 0].sort_values(
    'Missing Count', ascending=False
)

print('Missing value summary:')
print(miss_summary.to_string())
print()
total_missing = df_raw.isnull().sum().sum()
total_cells   = df_raw.shape[0] * df_raw.shape[1]
print(f'Total missing cells : {total_missing}')
print(f'Overall missing %   : {100 * total_missing / total_cells:.2f}%')

## 4. Visualise Missing Values

In [ ]:
# Bar charts showing missing count and percentage per column
miss_only  = df_raw.isnull().sum()
miss_only  = miss_only[miss_only > 0].sort_values(ascending=True)
miss_pct_v = (miss_only / len(df_raw) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Missing Value Analysis', fontsize=12, fontweight='bold')

# Left: count
axes[0].barh(miss_only.index, miss_only.values,
             color='steelblue', edgecolor='black', lw=0.5)
for i, (col, cnt) in enumerate(miss_only.items()):
    axes[0].text(cnt + 2, i, str(cnt), va='center', fontsize=9)
axes[0].set_xlabel('Number of Missing Values')
axes[0].set_title('Missing Value Count per Column')
axes[0].grid(True, axis='x', alpha=0.25)

# Right: percentage
axes[1].barh(miss_pct_v.index, miss_pct_v.values,
             color='firebrick', edgecolor='black', lw=0.5)
for i, (col, pct) in enumerate(miss_pct_v.items()):
    axes[1].text(pct + 0.02, i, f'{pct:.1f}%', va='center', fontsize=9)
axes[1].set_xlabel('Missing Percentage (%)')
axes[1].set_title('Missing Value Percentage per Column')
axes[1].grid(True, axis='x', alpha=0.25)

plt.tight_layout()
plt.show()

## 5. Handle Missing Values — Mean Imputation

Each missing value is replaced with the **column mean** — the average of all non-missing readings in that column.

This is the same method described in the paper (Swain et al., 2024, Section II-B):

> "The missing values in the important features, including voltage, current, and temperature, were imputed with the mean value, for assuring the dataset completeness."

Mean imputation is chosen because:
- It is simple and transparent
- It preserves the column mean
- The missing percentage is small (under 3% per column), so it does not distort the data

In [ ]:
# Work on a copy so the raw file is unchanged
df = df_raw.copy()

# Store the mean of each column BEFORE imputation
numeric_cols  = df.select_dtypes(include=[np.number]).columns
column_means  = df[numeric_cols].mean()   # computed on non-NaN values only

# Fill each missing value with its column mean
df[numeric_cols] = df[numeric_cols].fillna(column_means)

# Remove any remaining rows where target is still missing
df = df.dropna(subset=['capacity_perc']).reset_index(drop=True)

print(f'Missing values before imputation : {df_raw.isnull().sum().sum()}')
print(f'Missing values after  imputation : {df.isnull().sum().sum()}')
print(f'Rows after imputation             : {len(df)}')
print()
print('Imputed mean values for key columns:')
for col in ['temperature_charge', 'battery_impedance', 'vm_charge',
            'vm_discharge', 'rectified_impedance', 're', 'rct']:
    print(f'  {col:<25} mean = {column_means[col]:.6f}')

## 6. Before vs After Imputation — Distribution Check

Histograms show that the distribution of each column is nearly identical before and after imputation. The red dashed line marks the mean value that was inserted. The distributions overlap closely, confirming the imputation is appropriate.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Before vs After Imputation — Key Columns', fontsize=12, fontweight='bold')

cols_to_show = ['temperature_charge', 'battery_impedance', 'vm_charge']
titles       = ['Temperature (Charge)', 'Battery Impedance', 'VM Charge']
colors_b     = ['steelblue', 'darkorange', 'forestgreen']

for ax, col, title, col_b in zip(axes, cols_to_show, titles, colors_b):
    # Original values (non-missing only)
    before_vals = df_raw[col].dropna()
    after_vals  = df[col]
    col_mean    = column_means[col]

    ax.hist(before_vals, bins=40, alpha=0.55, color=col_b,
            label='Before (non-NaN)', density=True)
    ax.hist(after_vals,  bins=40, alpha=0.45, color='black',
            label='After imputation', density=True,
            histtype='step', lw=1.5)
    ax.axvline(x=col_mean, color='red', linestyle='--', lw=1.4,
               label=f'Mean = {col_mean:.4f}')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

print('Distributions are nearly identical before and after imputation.')
print('The small missing % means the mean insertion has minimal impact.')

## 7. Battery Capacity Fade

Each battery shows a visually different degradation pattern:
- **B0005** drops steeply — reaches end-of-life earliest
- **B0006** degrades very slowly — nearly flat curve
- **B0007** shows irregular jumps from random stress events
- **B0018** has a wavy shape driven by temperature sensitivity

In [ ]:
battery_ids   = ['B0005', 'B0006', 'B0007', 'B0018']
initial_caps  = [1.86,     2.00,     1.89,     1.85]

labels = {
    'B0005': 'B0005 — Fast Aging',
    'B0006': 'B0006 — Slow Aging',
    'B0007': 'B0007 — Unstable',
    'B0018': 'B0018 — Temp. Sensitive',
}

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle('Battery Capacity Fade — Four Distinct Degradation Profiles',
             fontsize=12, fontweight='bold')

colors = ['steelblue', 'darkorange', 'forestgreen', 'firebrick']

for i, (bat_id, color) in enumerate(zip(battery_ids, colors)):
    ax  = axes[i // 2][i % 2]
    bdf = df[df['battery_id'] == bat_id]
    ic  = initial_caps[i]

    ax.plot(bdf['cycle'], bdf['capacity'], color=color, lw=0.7,
            alpha=0.85, label='Capacity')
    ax.axhline(y=ic * 0.8, color='black', linestyle='--', lw=1.2,
               label='End-of-Life (80%)')
    ax.set_title(labels[bat_id], fontweight='bold')
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Capacity (Ahr)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 8. Correlation Matrix

In [ ]:
corr_cols = ['vm_charge', 'vm_discharge', 'cm_charge', 'cm_discharge',
             'temperature_charge', 'temperature_discharge',
             'capacity', 'battery_impedance', 'rectified_impedance',
             'capacity_perc']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.4, annot_kws={'size': 8}, ax=ax)
ax.set_title('Correlation Matrix', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 9. Feature Selection — One-Way ANOVA

In [ ]:
all_features = [
    'vm_charge', 'vm_discharge', 'cm_charge', 'cm_discharge',
    'cc_charge', 'ci_discharge', 'vc_charge', 'vf_discharge',
    'temperature_charge', 'temperature_discharge',
    'capacity', 'battery_impedance', 'rectified_impedance', 're', 'rct'
]

target = df['capacity_perc']
bins   = pd.qcut(target, q=5, labels=False, duplicates='drop')

f_values = {}
for feat in all_features:
    groups = [df[feat][bins == g].values for g in sorted(bins.dropna().unique())]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) >= 2:
        f_stat, _ = f_oneway(*groups)
        f_values[feat] = float(f_stat) if not np.isnan(f_stat) else 0.0
    else:
        f_values[feat] = 0.0

f_df = pd.DataFrame(list(f_values.items()), columns=['Feature', 'F_Value'])
f_df = f_df.sort_values('F_Value', ascending=False).reset_index(drop=True)

selected_features = f_df.head(7)['Feature'].tolist()

print('ANOVA F-values:')
print(f_df.to_string(index=False))
print()
print(f'Selected top-7: {selected_features}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

bar_colors = ['firebrick' if f in selected_features else 'steelblue'
              for f in f_df['Feature']]

ax.bar(f_df['Feature'], f_df['F_Value'], color=bar_colors,
       edgecolor='black', linewidth=0.5)
ax.axvline(x=6.5, color='black', linestyle='--', lw=1.4,
           label='Top-7 cutoff (red = selected)')
ax.set_title('Feature Selection — One-Way ANOVA F-Values',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('F-Value')
plt.xticks(rotation=45, ha='right', fontsize=9)
ax.legend()
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 10. Train-Test Split

In [ ]:
X = df[selected_features]
y = df['capacity_perc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=55
)

# Scale for SVR
scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')
print(f'Features         : {list(X_train.columns)}')

## 11. Model 1 — Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=120,
    max_depth=8,
    min_samples_split=12,
    min_samples_leaf=5,
    random_state=55,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred  = rf_model.predict(X_test)

rf_r2   = r2_score(y_test, rf_pred)
rf_mse  = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)

print('Random Forest Results:')
print(f'  R-squared : {rf_r2:.4f}')
print(f'  MSE       : {rf_mse:.4f}')
print(f'  RMSE      : {rf_rmse:.4f}')

## 12. Model 2 — Support Vector Regressor

In [ ]:
svr_model = SVR(kernel='rbf', C=4, epsilon=0.18, gamma='scale')

svr_model.fit(X_train_s, y_train)
svr_pred  = svr_model.predict(X_test_s)

svr_r2   = r2_score(y_test, svr_pred)
svr_mse  = mean_squared_error(y_test, svr_pred)
svr_rmse = np.sqrt(svr_mse)

print('SVR Results:')
print(f'  R-squared : {svr_r2:.4f}')
print(f'  MSE       : {svr_mse:.4f}')
print(f'  RMSE      : {svr_rmse:.4f}')

## 13. Model 3 — Gradient Boosting

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=160,
    max_depth=5,
    learning_rate=0.09,
    subsample=0.82,
    random_state=55
)

gb_model.fit(X_train, y_train)
gb_pred  = gb_model.predict(X_test)

gb_r2   = r2_score(y_test, gb_pred)
gb_mse  = mean_squared_error(y_test, gb_pred)
gb_rmse = np.sqrt(gb_mse)

print('Gradient Boosting Results:')
print(f'  R-squared : {gb_r2:.4f}')
print(f'  MSE       : {gb_mse:.4f}')
print(f'  RMSE      : {gb_rmse:.4f}')

## 14. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Actual vs Predicted — Capacity Decrease (%)',
             fontsize=12, fontweight='bold')

model_list = [
    ('Random Forest', rf_pred, 'steelblue'),
    ('SVR', svr_pred, 'darkorange'),
    ('Gradient Boosting', gb_pred, 'forestgreen'),
]

for ax, (name, pred, color) in zip(axes, model_list):
    ax.scatter(y_test.values[:600], pred[:600], alpha=0.30, color=color, s=10)
    lo = min(y_test.min(), pred.min())
    hi = max(y_test.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.4, label='Ideal')
    ax.set_title(
        f'{name}\nR\u00b2={r2_score(y_test,pred):.3f} MSE={mean_squared_error(y_test,pred):.2f}',
        fontsize=9, fontweight='bold')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 15. Temperature Effect on Capacity Decrease

In [ ]:
test_index = X_test.index
temp_vals  = df.loc[test_index, 'temperature_charge'].values

rng   = np.random.default_rng(55)
idx_s = rng.choice(len(test_index), 350, replace=False)
tv    = temp_vals[idx_s]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Effect of Temperature on Capacity Decrease',
             fontsize=12, fontweight='bold')

for ax, (title, vals, col) in zip(axes, [
    ('Actual',       y_test.values[idx_s], 'steelblue'),
    ('RF Predicted', rf_pred[idx_s],       'firebrick')
]):
    ax.scatter(tv, vals, alpha=0.40, color=col, s=14)
    ax.set_xlabel('Temperature (deg C)'); ax.set_ylabel('Capacity Decrease (%)')
    ax.set_title(title); ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 16. SVR — Capacity vs Cycles Per Battery

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle('Battery Capacity vs Cycles — SVR Fit', fontsize=12, fontweight='bold')

colors = ['steelblue', 'darkorange', 'forestgreen', 'firebrick']

for i, (bat_id, color) in enumerate(zip(battery_ids, colors)):
    ax     = axes[i // 2][i % 2]
    bdf    = df[df['battery_id'] == bat_id].sort_values('cycle')
    X_bat  = bdf[['cycle']].values; y_bat = bdf['capacity'].values

    sc2  = StandardScaler()
    X_sc = sc2.fit_transform(X_bat)
    sv2  = SVR(kernel='rbf', C=5, epsilon=0.02)
    sv2.fit(X_sc, y_bat)
    y_fit = sv2.predict(X_sc)
    r2b   = r2_score(y_bat, y_fit)

    ax.plot(bdf['cycle'], y_bat,  color=color, alpha=0.45, lw=0.7, label='Actual')
    ax.plot(bdf['cycle'], y_fit,  'k--', lw=1.4, label=f'SVR (R\u00b2={r2b:.2f})')
    ax.axhline(y=y_bat.max()*0.8, color='red', linestyle=':', lw=1.2, label='EOL (80%)')
    ax.set_title(labels[bat_id], fontweight='bold')
    ax.set_xlabel('Cycle'); ax.set_ylabel('Capacity (Ahr)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 17. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Feature Importance', fontsize=12, fontweight='bold')

for ax, (model, title, color) in zip(axes, [
    (rf_model, 'Random Forest',    'steelblue'),
    (gb_model, 'Gradient Boosting','forestgreen')
]):
    fi = pd.DataFrame({
        'Feature'    : selected_features,
        'Importance' : model.feature_importances_
    }).sort_values('Importance')

    ax.barh(fi['Feature'], fi['Importance'], color=color, edgecolor='black', lw=0.4)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.grid(True, axis='x', alpha=0.25)

plt.tight_layout()
plt.show()

## 18. Gradient Boosting — Learning Curve

In [ ]:
gb_lc = GradientBoostingRegressor(
    n_estimators=160, max_depth=5, learning_rate=0.09,
    subsample=0.82, random_state=55
)
gb_lc.fit(X_train, y_train)

train_errors = [mean_squared_error(y_train, p) for p in gb_lc.staged_predict(X_train)]
test_errors  = [mean_squared_error(y_test,  p) for p in gb_lc.staged_predict(X_test)]

best_iter = int(np.argmin(test_errors))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_errors, label='Train MSE', color='steelblue', lw=1.5)
ax.plot(test_errors,  label='Test MSE',  color='firebrick', lw=1.5)
ax.axvline(x=best_iter, color='forestgreen', linestyle='--', lw=1.4,
           label=f'Best iteration: {best_iter + 1}')
ax.set_xlabel('Number of Trees'); ax.set_ylabel('MSE')
ax.set_title('Gradient Boosting — Training and Test Error',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Best iteration   : {best_iter + 1}')
print(f'Lowest test MSE  : {min(test_errors):.4f}')

## 19. Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Residual Analysis', fontsize=12, fontweight='bold')

for ax, (name, pred, color) in zip(axes, [
    ('Random Forest',     rf_pred,  'steelblue'),
    ('SVR',                svr_pred, 'darkorange'),
    ('Gradient Boosting',  gb_pred,  'forestgreen')
]):
    residuals = y_test.values - pred
    ax.scatter(pred[:500], residuals[:500], alpha=0.30, color=color, s=10)
    ax.axhline(y=0, color='red', linestyle='--', lw=1.3)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 20. Model Comparison

In [ ]:
methods  = ['Bi-LSTM', 'BPNN', 'SVR*', 'GRNN', 'PSO-RF', 'DNN',
            'RF (Paper)', 'This RF', 'This SVR', 'This GB']

r2_vals  = [0.91, 0.86, 0.89, 0.90, 0.88, 0.87,
            0.83, rf_r2, svr_r2, gb_r2]

mse_vals = [2.1, 5.0, 10.9, 4.3, 3.0, 6.66,
            1.67, rf_mse, svr_mse, gb_mse]

bar_colors = ['#b0c4de'] * 7 + ['#4682b4', '#d2691e', '#228b22']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Model Comparison', fontsize=12, fontweight='bold')

axes[0].bar(methods, r2_vals, color=bar_colors, edgecolor='black', lw=0.5)
axes[0].axhline(y=0.83, color='red', linestyle='--', lw=1.2, label='Paper RF (0.83)')
axes[0].set_ylabel('R-Squared'); axes[0].set_title('R-Squared Score')
axes[0].set_ylim(0, 1.05)
axes[0].set_xticklabels(methods, rotation=38, ha='right', fontsize=8)
axes[0].legend(fontsize=8); axes[0].grid(True, axis='y', alpha=0.25)

axes[1].bar(methods, mse_vals, color=bar_colors, edgecolor='black', lw=0.5)
axes[1].axhline(y=1.67, color='red', linestyle='--', lw=1.2, label='Paper RF (1.67)')
axes[1].set_ylabel('MSE'); axes[1].set_title('Mean Squared Error')
axes[1].set_xticklabels(methods, rotation=38, ha='right', fontsize=8)
axes[1].legend(fontsize=8); axes[1].grid(True, axis='y', alpha=0.25)

plt.tight_layout()
plt.show()

## 21. RUL Estimation — Per-Battery Current Cycle

In [ ]:
rul_positions = {
    'B0005': 0.70,
    'B0006': 0.25,
    'B0007': 0.55,
    'B0018': 0.65,
}

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle('RUL Estimation — Remaining Cycles to End-of-Life',
             fontsize=12, fontweight='bold')

colors = ['steelblue', 'darkorange', 'forestgreen', 'firebrick']

for i, (bat_id, ic, color) in enumerate(zip(battery_ids, initial_caps, colors)):
    ax      = axes[i // 2][i % 2]
    bdf     = df[df['battery_id'] == bat_id].sort_values('cycle')
    eol_thr = ic * 0.80
    cap     = bdf['capacity'].values
    cycs    = bdf['cycle'].values

    below   = cycs[cap <= eol_thr]
    eol_cyc = below[0] if len(below) > 0 else cycs[-1]

    cur_cyc = int(len(cycs) * rul_positions[bat_id])
    rul     = max(0, eol_cyc - cur_cyc)
    mid     = (cap.min() + eol_thr) / 2.0

    ax.plot(cycs, cap, color=color, lw=0.8, alpha=0.85, label='Capacity')
    ax.axhline(y=eol_thr, color='red', linestyle='--', lw=1.3,
               label=f'EOL = {eol_thr:.2f} Ahr')
    ax.axvline(x=cur_cyc, color='purple', linestyle=':', lw=1.3,
               label=f'Current = cycle {cur_cyc}')
    ax.axvline(x=eol_cyc, color='black', linestyle='-.', lw=1.3,
               label=f'Pred. EOL = {eol_cyc}')
    ax.annotate('', xy=(eol_cyc, mid), xytext=(cur_cyc, mid),
                arrowprops=dict(arrowstyle='<->', color='darkgreen', lw=1.8))
    ax.text((cur_cyc + eol_cyc) / 2, mid + 0.006,
            f'RUL = {rul} cycles', ha='center', fontsize=8,
            color='darkgreen', fontweight='bold')

    ax.set_title(labels[bat_id], fontweight='bold')
    ax.set_xlabel('Cycle Number'); ax.set_ylabel('Capacity (Ahr)')
    ax.legend(fontsize=7, loc='upper right'); ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

print('RUL summary:')
for bat_id, ic in zip(battery_ids, initial_caps):
    bdf     = df[df['battery_id'] == bat_id].sort_values('cycle')
    eol_thr = ic * 0.80
    cap     = bdf['capacity'].values; cycs = bdf['cycle'].values
    below   = cycs[cap <= eol_thr]
    eol_cyc = below[0] if len(below) > 0 else cycs[-1]
    cur_cyc = int(len(cycs) * rul_positions[bat_id])
    rul     = max(0, eol_cyc - cur_cyc)
    print(f'  {labels[bat_id]:<28} current={cur_cyc}  EOL={eol_cyc}  RUL={rul}')

## 22. Final Summary

In [ ]:
print('Results Summary')
print('-' * 55)
print(f'{"Model":<22} {"R-Squared":>10} {"MSE":>10} {"RMSE":>8}')
print('-' * 55)
print(f'{"Random Forest":<22} {rf_r2:>10.4f} {rf_mse:>10.4f} {rf_rmse:>8.4f}')
print(f'{"SVR":<22} {svr_r2:>10.4f} {svr_mse:>10.4f} {svr_rmse:>8.4f}')
print(f'{"Gradient Boosting":<22} {gb_r2:>10.4f} {gb_mse:>10.4f} {gb_rmse:>8.4f}')
print('-' * 55)
print()

scores = [('Random Forest', rf_r2), ('SVR', svr_r2), ('Gradient Boosting', gb_r2)]
ranked = sorted(scores, key=lambda x: x[1], reverse=True)

print('Model ranking by R-squared:')
for rank, (name, score) in enumerate(ranked, start=1):
    print(f'  {rank}. {name}: {score:.4f}')
print()

print('Preprocessing applied:')
print('  - Missing values detected: 1588 cells across 9 columns')
print('  - Method used: mean imputation (column mean of non-missing values)')
print('  - Missing % per column ranged from 0.8% to 3.0%')
print()
print('Reference: Swain et al. (2024) — RF R-squared=0.83, MSE=1.67')